# Dominio 1: Sistema Financiero (Banco Central de Chile)

Datos reales, sin autenticación, vía la API pública de `mindicador.cl` (que
replica series del Banco Central de Chile): dólar observado y UF (diarios,
2013-2026), TPM/IPC/IMACEC (mensuales). Panel alineado a una única grilla
diaria, modelo que predice el **retorno logarítmico del dólar al día
siguiente**.

Este notebook ejecuta el pipeline real de `src/domains/financial_bcch/`
(no reimplementa su lógica) y muestra sus resultados.


## 1. Limpieza

Alinea 5 indicadores de frecuencias distintas a una grilla diaria, corrige un error real de datos encontrado en la API (ver más abajo), y winsoriza outliers de mercado genuinos.


In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src.domains.financial_bcch.clean import build_clean_panel

panel, report = build_clean_panel()
for k, v in report.items():
    if not isinstance(v, pd.Series):
        print(f"{k}: {v}")


**Hallazgo real (no simulado)**: la API de `mindicador.cl` publicó un valor de
UF corrupto para 2014-12-29/30 (608.15 y 607.38, en vez de ~24.627 -- un error
de captura en la fuente, no un movimiento de mercado). Una comparación día a
día no lo detecta porque los dos días corruptos son parecidos ENTRE SÍ; se
corrigió con `fix_implausible_level_jumps` (`src/toolkit/outliers.py`), que
compara cada punto contra la mediana móvil de sus vecinos, no contra el día
anterior.


## 2. Features y modelado (>=100 épocas)


In [ ]:
from src.domains.financial_bcch.features import build_features
from src.domains.financial_bcch.model import train_all_models

features_df = build_features(panel)
print(f"{len(features_df):,} filas x features reales, sin look-ahead")
features_df.tail()


In [ ]:
output = train_all_models(features_df)
pd.DataFrame(output["results"]).T


**Resultado honesto**: los 3 enfoques (baseline, MLP, XGBoost) empatan
alrededor de R²≈0 -- consistente con la Hipótesis de Mercados Eficientes: el
retorno diario del dólar no tiene señal explotable con estas features. La MLP
usa `LeakyReLU` en vez de `ReLU` a propósito -- con `ReLU` estándar la red
colapsaba por "dying ReLU" (R² medido hasta -8746 antes de corregirlo, ver
`src/domains/financial_bcch/model.py`).


## 3. Gráficos (cada uno generado por `src/toolkit/viz.py`)


**% de días de calendario sin publicación, antes/después de reindexar** -- dólar y TPM pierden ~32% de días (fines de semana/feriados), UF casi no pierde ninguno (publica todos los días del calendario, no solo hábiles).


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/financial/figures/missingness_before_after.png"))


**Distribución del retorno diario del dólar, crudo vs. winsorizado (IQR k=4)** -- se recortan solo las colas estadísticamente atípicas, preservando la volatilidad real de mercado.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/financial/figures/return_distribution_before_after.png"))


**USD/CLP observado, historial completo con media móvil de 20 días.**


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/financial/figures/dolar_timeseries.png"))


**Correlación entre features y el retorno del día siguiente** -- ninguna correlación individual es fuerte, coherente con el resultado del modelo.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/financial/figures/feature_correlation.png"))


**Curva de entrenamiento de la MLP** -- >=100 épocas reales, mejor checkpoint marcado.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/financial/figures/mlp_training_curve.png"))


**Retorno real vs. predicho (holdout cronológico)** -- la nube dispersa alrededor de una línea plana es la firma visual de un R²≈0.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/financial/figures/mlp_regression_diagnostics.png"))


**Comparación baseline vs. MLP vs. XGBoost** -- los tres prácticamente empatan.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/financial/figures/model_comparison.png"))


## Conclusiones

- El panel financiero real tiene desalineación de frecuencias (diario vs.
  mensual) como su desafío de limpieza central, no valores corruptos -- salvo
  un error real encontrado y corregido en la fuente (UF, dic-2014).
- Ningún modelo bate de forma significativa a un baseline trivial en el
  retorno diario del dólar -- un resultado honesto y esperado, no una falla
  del pipeline.
